# Maximum Likelihood Classification

This notebook performs pixel-based Maximum Likelihood Classification (MLC) of the processed WorldView-3 multispectral image using user-provided training samples.

The workflow assumes that the training samples are already available as a polygon vector layer (e.g., ESRI Shapefile or GeoPackage). Since the creation of training samples depends on the software chosen by the user (ENVI, QGIS, ArcGIS, etc.), this step is not included in the workflow.

In this work, the Regions of Interest (ROIs) were manually delineated in ENVI and subsequently converted to a polygon vector layer before classification.

### Input

- Masked multispectral image (`VNIR_SWIR_buildings.tif`)
- Training polygons containing a field named `class`

### Output

- Classified raster (`MLC_classification.tif`)

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio

from functions.classification import *


# Input and output files

data_dir = Path("../output")
training_dir = Path("../data")

raster_path = data_dir / "VNIR_SWIR_buildings.tif"
training_path = training_dir / "training_samples.shp"

output_path = data_dir / "MLC_classification.tif"

class_field = "class"


# Load raster and training data

training_gdf = gpd.read_file(training_path)

src = rasterio.open(raster_path)
image = src.read()
profile = src.profile


# Extract training samples

X, y = extract_training_samples(
    src,
    training_gdf,
    class_field=class_field,
)


# Estimate class statistics

classes_info = estimate_class_statistics(
    X,
    y,
    add_cov_epsilon=1e-6,
)


# Perform Maximum Likelihood Classification

classified = maximum_likelihood_classifier(
    image=image,
    classes_info=classes_info,
    nodata_value=src.nodata,
)



# Convert class labels to integer values

class_labels = list(classes_info.keys())
class_to_int = {cls: i + 1 for i, cls in enumerate(class_labels)}

classified_int = np.zeros(classified.shape, dtype=np.uint8)

for cls, value in class_to_int.items():
    classified_int[classified == cls] = value



# Save classified raster

profile.update(
    count=1,
    dtype="uint8",
    nodata=0,
)

with rasterio.open(output_path, "w", **profile) as dst:
    dst.write(classified_int, 1)

print(f"Classification saved as:\n{output_path.name}")